# JARVIS · Ajuste fino con LoRA (Etapa 12)

Ajusta **Qwen3-4B-Instruct-2507** (el modelo de `jarvis:base`) para que hable con el estilo de JARVIS: de usted, breve, sin citas inventadas.

**Antes de empezar**
1. En su PC: `.venv\Scripts\python -m scripts.export_dataset estilo-jarvis --destino nube`
   → crea `training/datasets/estilo-jarvis-nube.jsonl`.
2. Aquí en Colab: **Entorno de ejecución › Cambiar tipo de entorno › GPU T4** (gratis).
3. Ejecute las celdas en orden (▶). Tarda ~20–30 min.

**Privacidad:** este cuaderno sólo recibe el dataset de estilo, generado a partir de preguntas genéricas. `export_dataset` **se niega** a exportar para la nube los datasets con datos personales (por ejemplo las correcciones 👎).

**Al final** se descarga `jarvis-v1.Q4_K_M.gguf` (~2,5 GB). En su PC:
```
.venv\Scripts\python -m scripts.models importar models\jarvis-v1.Q4_K_M.gguf --tag jarvis:v1
.venv\Scripts\python -m scripts.compare_finetune --nuevo jarvis:v1
```

## 1. Instalar Unsloth (entrena 2× más rápido y con menos memoria)

In [ ]:
%%capture
!pip install unsloth

## 2. Subir el dataset (`estilo-jarvis-nube.jsonl`)

In [ ]:
from google.colab import files
import json

subido = files.upload()  # elija training/datasets/estilo-jarvis-nube.jsonl
nombre = next(iter(subido))
ejemplos = [json.loads(linea) for linea in subido[nombre].decode("utf-8").splitlines() if linea.strip()]
print(len(ejemplos), "ejemplos")
print(ejemplos[0]["messages"][1]["content"], "→", ejemplos[0]["messages"][2]["content"][:120])

## 3. Cargar el modelo en 4 bits y prepararlo para LoRA

- **QLoRA:** el modelo queda congelado en 4 bits; sólo se entrenan matrices pequeñas (r=16) añadidas a las capas de atención y MLP. Cabe en la T4 de 15 GB.
- `max_seq_length` 2048: el SYSTEM de JARVIS (~700 tokens) + pregunta + respuesta caben con holgura.

In [ ]:
from unsloth import FastLanguageModel

modelo, tokenizer = FastLanguageModel.from_pretrained(
    model_name="unsloth/Qwen3-4B-Instruct-2507",
    max_seq_length=2048,
    load_in_4bit=True,
)
modelo = FastLanguageModel.get_peft_model(
    modelo,
    r=16,
    lora_alpha=16,
    lora_dropout=0,
    target_modules=["q_proj", "k_proj", "v_proj", "o_proj", "gate_proj", "up_proj", "down_proj"],
    use_gradient_checkpointing="unsloth",
    random_state=3407,
)

## 4. Dar formato de chat y separar 10 % para validar

In [ ]:
from datasets import Dataset

def a_texto(ejemplo):
    return {"text": tokenizer.apply_chat_template(ejemplo["messages"], tokenize=False)}

datos = Dataset.from_list(ejemplos).map(a_texto).train_test_split(test_size=0.1, seed=3407)
print(datos)

## 5. Entrenar

- Sólo aprende de las **respuestas** de JARVIS (`train_on_responses_only`), no del SYSTEM ni de las preguntas.
- 2 épocas y tasa de aprendizaje baja: ajustar el estilo sin olvidar lo que el modelo ya sabe.

In [ ]:
from trl import SFTConfig, SFTTrainer
from unsloth.chat_templates import train_on_responses_only

entrenador = SFTTrainer(
    model=modelo,
    tokenizer=tokenizer,
    train_dataset=datos["train"],
    eval_dataset=datos["test"],
    args=SFTConfig(
        dataset_text_field="text",
        per_device_train_batch_size=2,
        gradient_accumulation_steps=4,
        num_train_epochs=2,
        learning_rate=1e-4,
        warmup_ratio=0.05,
        lr_scheduler_type="cosine",
        logging_steps=5,
        eval_strategy="epoch",
        optim="adamw_8bit",
        weight_decay=0.01,
        seed=3407,
        output_dir="salidas",
        report_to="none",
    ),
)
entrenador = train_on_responses_only(
    entrenador,
    instruction_part="<|im_start|>user\n",
    response_part="<|im_start|>assistant\n",
)
estadisticas = entrenador.train()
print(estadisticas)
print(entrenador.evaluate())

## 6. Probarlo antes de exportar (¿responde de usted?)

In [ ]:
FastLanguageModel.for_inference(modelo)
sistema = ejemplos[0]["messages"][0]["content"]
for pregunta in ["oye, ¿tú qué sabes hacer?", "Dame una idea para la cena", "Pon música"]:
    mensajes = [{"role": "system", "content": sistema}, {"role": "user", "content": pregunta}]
    entrada = tokenizer.apply_chat_template(mensajes, add_generation_prompt=True, return_tensors="pt").to("cuda")
    salida = modelo.generate(input_ids=entrada, max_new_tokens=120, temperature=0.5, do_sample=True)
    print("›", pregunta)
    print(tokenizer.decode(salida[0][entrada.shape[1]:], skip_special_tokens=True), "\n")

## 7. Exportar a GGUF Q4_K_M y descargar

Mismo tamaño y cuantización que `jarvis:base`: cabe igual en la GTX 1650 Ti (100 % en la GPU).

In [ ]:
modelo.save_pretrained_gguf("jarvis-v1", tokenizer, quantization_method="q4_k_m")
import glob, os, shutil
gguf = max(glob.glob("jarvis-v1*/*.gguf") + glob.glob("*.gguf"), key=os.path.getsize)
shutil.move(gguf, "jarvis-v1.Q4_K_M.gguf")
print(os.path.getsize("jarvis-v1.Q4_K_M.gguf") / 1e9, "GB")
files.download("jarvis-v1.Q4_K_M.gguf")

## 8. En su PC

1. Guarde el archivo en `models\` (esa carpeta no va a Git).
2. `.venv\Scripts\python -m scripts.models importar models\jarvis-v1.Q4_K_M.gguf --tag jarvis:v1`
3. `.venv\Scripts\python -m scripts.compare_finetune --nuevo jarvis:v1`
4. Si gana: `.venv\Scripts\python -m scripts.models activar jarvis:v1` · si no: no pasa nada, `jarvis:base` sigue activo (y `scripts.models volver` deshace una activación).